把碎片离子放在同一个sheet中
但是有一个bug在于：样本量非常大的时候，例如有300个样本，那么就得有600列，非常不便于处理和可视化

In [14]:
import pandas as pd
import os
import numpy as np
import re
# 主要Excel文件路径
main_excel_path = 'Y:/PHD/2024.1.16乳腺癌最新数据整理/data.xlsx'
main_df = pd.read_excel(main_excel_path)

# 样品Excel文件夹路径
folder_path = 'Y:/PHD/2024.1.16乳腺癌最新数据整理/2sampleTest'

# 近似匹配函数
def is_close(a, b, tolerance=0.2):
    try:
        a = float(a)
        b = float(b)
    except ValueError:
        return False
    return abs(a - b) <= tolerance

# 处理每个样品文件
for file in os.listdir(folder_path):
    if file.endswith('.xlsx') or file.endswith('.xls'):
        sample_df = pd.read_excel(os.path.join(folder_path, file))
        # 通过正则表达式确定 mz 和 intensity 列
        mz_cols = [col for col in sample_df.columns if re.match(r'mz\d+$', col)]
        intensity_cols = ['intensity' + re.findall(r'\d+', col)[0] for col in mz_cols if 'intensity' + re.findall(r'\d+', col)[0] in sample_df.columns]

        for index, row in sample_df.iterrows():
            material_name = row['lipid_omega_name']
            matching_rows = main_df[main_df['双键类别'] == material_name]

            if not matching_rows.empty:
                for mz_col, int_col in zip(mz_cols, intensity_cols):
                    mz_value = row.get(mz_col)
                    if mz_value and pd.notna(mz_value):
                        for ion_col in ['碎片离子1', '碎片离子2']:
                            main_ion_value = matching_rows[ion_col].iloc[0]
                            if pd.notna(main_ion_value) and is_close(mz_value, main_ion_value):
                                # 查找或创建新列
                                new_col_name = ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')
                                if new_col_name not in main_df.columns:
                                    main_df[new_col_name] = np.nan
                                main_df.loc[matching_rows.index, new_col_name] = row[int_col]
                        # 如果没有找到匹配的物质名称
            else:
                main_df.loc[main_df['双键类别'] == material_name, ['碎片离子1', '碎片离子2']] = np.nan



# 保存更新后的DataFrame到新的Excel文件
main_df.to_excel('updated_main_excel.xlsx', index=False)

将结果分别保存到不同的 Excel 工作表

In [13]:
import pandas as pd
import os
import re

# 主要Excel文件路径
#####
main_excel_path = 'C:/Users/Administrator/Desktop/data.xlsx' 
main_df = pd.read_excel(main_excel_path)

# 样品Excel文件夹路径
######
folder_path = 'C:/Users/Administrator/Desktop/20240305 shengjing'

# 近似匹配函数
def is_close(a, b, tolerance=0.2):
    try:
        a = float(a)
        b = float(b)
    except ValueError:
        return False
    return abs(a - b) <= tolerance

# 创建用于不同工作表的DataFrame
df_ion1 = main_df.copy()
df_ion2 = main_df.copy()

# 处理每个样品文件
for file in os.listdir(folder_path):
    if file.endswith('.xlsx') or file.endswith('.xls'):
        sample_df = pd.read_excel(os.path.join(folder_path, file))
        # 通过正则表达式确定 mz 和 intensity 列
        mz_cols = [col for col in sample_df.columns if re.match(r'mz\d+$', col)]
        intensity_cols = ['intensity' + re.findall(r'\d+', col)[0] for col in mz_cols if 'intensity' + re.findall(r'\d+', col)[0] in sample_df.columns]

        for index, row in sample_df.iterrows():
            material_name = row['lipid_omega_name']
            matching_rows = main_df[main_df['双键类别'] == material_name]

            if not matching_rows.empty:
                for mz_col, int_col in zip(mz_cols, intensity_cols):
                    mz_value = row.get(mz_col)
                    if mz_value and pd.notna(mz_value):
                        for ion_col in ['碎片离子1', '碎片离子2']:
                            main_ion_value = matching_rows[ion_col].iloc[0]
                            if pd.notna(main_ion_value) and is_close(mz_value, main_ion_value):
                                # 根据ion_col更新不同的DataFrame
                                if ion_col == '碎片离子1':
                                    df_ion1.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
                                elif ion_col == '碎片离子2':
                                    df_ion2.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]

# 保存结果到不同的工作表
#######
with pd.ExcelWriter('C:/Users/Administrator/Desktop/2024030501.xlsx') as writer:
    main_df.to_excel(writer, sheet_name='Main Data', index=False)
    df_ion1.to_excel(writer, sheet_name='Ion 1', index=False)
    df_ion2.to_excel(writer, sheet_name='Ion 2', index=False)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion1.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion2.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246

C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion1.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion2.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246

C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion1.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion2.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246

C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion1.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion2.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246

C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion1.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion2.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246

C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion1.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion2.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246

... (12 outputs left out)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion1.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion2.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246

C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion1.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion2.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246

C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion1.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion2.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246

C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion1.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ion2.loc[matching_rows.index, ion_col + '_' + file.replace('.xlsx', '').replace('.xls', '')] = row[int_col]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11452\3447233246

In [15]:
def combine_df_ions_to_existing_excel(df_ion1, df_ion2, some_n, existing_excel_path, new_sheet_name):
    """
    Combines specific columns of two DataFrames (df_ion1 and df_ion2) based on their suffixes
    and saves the combined data into a new sheet of an existing Excel file.

    :param df_ion1: First DataFrame to be combined.
    :param df_ion2: Second DataFrame to be combined.
    :param some_n: Number of initial columns to check for name matching.
    :param existing_excel_path: Path to the existing Excel file where the new sheet will be added.
    :param new_sheet_name: Name of the new sheet in the existing Excel file.
    """
    # 检查DataFrame的大小是否一致
    if df_ion1.shape != df_ion2.shape:
        raise ValueError("DataFrames 'df_ion1' and 'df_ion2' do not have the same shape.")
    
    # 检查DataFrame前几列的列名是否匹配
    if not df_ion1.columns[:some_n].equals(df_ion2.columns[:some_n]):
        raise ValueError("The first columns of the DataFrames 'df_ion1' and 'df_ion2' do not match.")
    
    # 将“碎片离子1”和“碎片离子2”带有相同后缀的列相加
    combined_data = df_ion1.iloc[:, :some_n]  # 前几列
    for col in df_ion1.columns[some_n:]:
        suffix = col[len("碎片离子1_"):]
        combined_col_name = "combined_" + suffix
        combined_data[combined_col_name] = df_ion1[col].fillna(0) + df_ion2["碎片离子2_" + suffix].fillna(0)
    
    # 将结果保存到现有的Excel文件中的新工作表
    with pd.ExcelWriter(existing_excel_path, mode='a', engine='openpyxl') as writer:
        combined_data.to_excel(writer, sheet_name=new_sheet_name, index=False)

    return combined_data  # 返回新的DataFrame以供检查

# 示例用法（请根据需要替换参数）
main_excel_path ='C:/Users/Administrator/Desktop/THEFIRSTBATCH_result.xlsx'
df_ion1 = pd.read_excel('C:/Users/Administrator/Desktop/THEFIRSTBATCH_result.xlsx',"Ion 1")
df_ion2 = pd.read_excel('C:/Users/Administrator/Desktop/THEFIRSTBATCH_result.xlsx',"Ion 2")
combine_df_ions_to_existing_excel(df_ion1, df_ion2, 5, 'C:/Users/Administrator/Desktop/THEFIRSTBATCH_result.xlsx', 'add')

,母离子,脂质种类,双键类别,碎片离子1,碎片离子2,combined_breast cancer before 1,combined_breast cancer before 10,combined_breast cancer before 11,combined_breast cancer before 12,combined_breast cancer before 13,...,combined_breast cancer before 55,combined_breast cancer before 56,combined_breast cancer before 57,combined_breast cancer before 58,combined_breast cancer before 59,combined_breast cancer before 6,combined_breast cancer before 60,combined_breast cancer before 7,combined_breast cancer before 8,combined_breast cancer before 9
0,790.5,PC 14:0_18:1,PC(14:0_18:1(Δ8)),608.5,634.5,252.549576,0.000000,105.000000,76.000000,61.000000,...,60.000000,0.000000,55.000000,0.000000,0.0,278.982147,76.000000,100.000000,0.000000,74.000000
1,NaN,NaN,PC(14:0_18:1(Δ9)),622.5,648.5,151.000000,471.989746,1203.662842,297.000000,485.114899,...,187.519684,208.705750,178.000000,96.234497,0.0,692.088135,186.000000,577.308472,567.781738,351.000000
2,NaN,NaN,PC(14:0_18:1(Δ11)),650.5,676.5,143.551727,1183.231812,2322.183777,898.912720,1081.464844,...,536.881104,279.849884,406.590942,236.248535,0.0,1771.924438,463.820770,1380.782837,354.000000,865.323730
3,804.5,PC 15:0_18:1,PC(15:0_18:1(Δ8)),622.5,648.5,11152.891602,2596.262695,3156.859375,1616.318481,0.000000,...,0.000000,1829.655518,0.000000,0.000000,0.0,2608.775635,1156.267578,2431.515625,2321.444580,0.000000
4,NaN,NaN,PC(15:0_18:1(Δ9)),636.5,662.5,343.555786,1593.863037,1083.508423,919.087891,704.843872,...,910.675049,337.345673,159.000000,550.632477,42.0,1593.574219,198.578522,1123.337402,732.767090,563.314819
5,NaN,NaN,PC(15:0_18:1(Δ11)),664.5,690.5,76.000000,116.000000,169.850281,180.334961,76.000000,...,102.000000,0.000000,0.000000,174.000000,0.0,128.620117,72.000000,160.000000,202.191162,337.954468
6,816.5,PC 16:1_18:1,PC(16:1(Δ9)_18:1(Δ8)),634.5,660.5,171.415222,195.850281,262.799301,165.970215,159.000000,...,97.000000,0.000000,104.000000,134.000000,0.0,239.000000,83.244934,199.040710,291.267242,311.849792
7,NaN,NaN,PC(16:1(Δ9)_18:1(Δ9)),648.5,674.5,211.213745,949.513306,657.775757,641.040344,1018.115295,...,239.318024,773.563477,442.379028,305.487793,0.0,764.210571,282.703064,1379.607727,1047.450012,811.023926
8,NaN,NaN,PC(16:1(Δ9)_18:1(Δ11)),676.5,702.5,23862.380768,85126.300659,72416.593262,95459.945312,53176.668457,...,38016.337158,18917.406250,16911.571838,42621.925781,0.0,96540.731934,8531.166016,95457.868896,62213.067444,113004.677612
9,818.5,PC 16:0_18:1,PC(16:0_18:1(Δ8)),636.5,662.5,210.843079,136.651245,306.908142,291.349487,168.306274,...,141.199524,102.000000,114.121521,43.000000,0.0,141.000000,95.000000,140.721802,238.768494,148.000000
